In [ ]:
import pandas as pd
import numpy as np
import re
import networkx as nx
from matplotlib.lines import Line2D
import os
import cstarpy.inference
import cstarpy.integration
from cstarpy.preprocessing import PerturbationMagnitude
import seaborn as sns
from matplotlib.colors import TwoSlopeNorm
import matplotlib.pyplot as plt
import pylab as plt
import openpyxl

In [ ]:

CD8_SZABO_FILE = "CD8_Szabo_Activation_Markers.csv"
COMPOUND_FILE  = "cd8_limma_merged_filtered_targets_ic50.csv"
cd8_szabo = pd.read_csv(CD8_SZABO_FILE)\
    .rename(columns={"Unnamed: 0": "ensembl", "avg_log2FC": "szabo_lfc"})\
    .dropna(subset=["gene", "szabo_lfc"])\
    .drop_duplicates(subset="gene")[["gene", "szabo_lfc"]]

szabo_lookup = cd8_szabo.set_index("gene")["szabo_lfc"]
print(f"CD8 Szabo genes: {len(cd8_szabo):,}")
df = pd.read_csv(COMPOUND_FILE)\
    .rename(columns={"Unnamed: 0": "sample", "Unnamed: 1": "row_idx"})

df_avg = (
    df.groupby(["compound_name", "gene"])
    .agg(logFC=("logFC", "mean"), adj_P_Val=("adj.P.Val", "mean"))
    .reset_index()
)
df_avg["logFC_thresh"] = np.where(df_avg["adj_P_Val"] < 0.05, df_avg["logFC"], 0)

print(f"Significant genes kept  : {(df_avg['adj_P_Val'] < 0.05).sum():,}")
print(f"Insignificant → zeroed  : {(df_avg['adj_P_Val'] >= 0.05).sum():,}")
pivot = df_avg.pivot_table(
    index   = "compound_name",
    columns = "gene",
    values  = "logFC_thresh",
    aggfunc = "first"
).fillna(0)
common_genes  = pivot.columns.intersection(szabo_lookup.index)
pivot_aligned = pivot[common_genes]
szabo_aligned = szabo_lookup[common_genes].values
dpd_vec = pivot_aligned.values @ szabo_aligned
dpd_df = pd.DataFrame({
    "compound_name" : pivot_aligned.index,
    "dpd"           : dpd_vec.round(4),
    "direction"     : ["drives_activation" if x > 0 else "drives_resting"
                     for x in dpd_vec]
}).sort_values("dpd", ascending=False).reset_index(drop=True)
targets = pd.read_csv("cd8_limma_merged_filtered_targets_ic50.csv")[
    ["compound_name", "target_protein", "mechanism"]
].drop_duplicates(subset="compound_name")
dpd_df = dpd_df.merge(targets, on="compound_name", how="left")
dpd_df.to_csv("dpd_sum_per_compound_raw.csv", index=False)


In [ ]:
# 4. Align to Szabo reference vector (raw) 

REF_COL = "log2FoldChange"

common_genes  = pivot.columns.intersection(cd8_szabo.index)
pivot_aligned = pivot[common_genes]
szabo_aligned = cd8_szabo.loc[common_genes, REF_COL].values

#  5. DPD per compound = pivot @ szabo (no normalisation)

dpd_vec = pivot_aligned.values @ szabo_aligned

dpd_df = pd.DataFrame({
    "compound_name" : pivot_aligned.index,
    "dpd"           : dpd_vec.round(4),
    "direction"     : ["drives_activation" if x > 0 else "drives_resting"
                       for x in dpd_vec]
}).sort_values("dpd", ascending=False).reset_index(drop=True)

#  6. Add targets and mechanism 

targets = pd.read_csv(COMPOUND_FILE)[
    ["compound_name", "target_protein", "mechanism"]
].drop_duplicates(subset="compound_name")

dpd_df = dpd_df.merge(targets, on="compound_name", how="left")

#  7. Save 
dpd_df.to_csv("dpd_sum_per_compound_raw_btla.csv", index=False)
print("\n✓ Saved dpd_sum_per_compound_raw_btla.csv")
